# Pipeline de avaliação — NER (BERTimbau + LeNER-Br), Automatic Judge e métricas

Código nos módulos `ner.py`, `juiz.py` e `metricas.py` (mesma pasta). Rode as células em ordem.

In [ ]:
# No Colab: clone o repositório e entre na pasta
# !git clone https://github.com/SEU_USUARIO/SEU_REPO.git
# %cd SEU_REPO

!pip install -q datasets evaluate setuptools_scm spacy
!pip install seqeval --no-build-isolation
!python -m spacy download pt_core_news_lg

## 1. NER

In [ ]:
from transformers import AutoTokenizer
import ner

ner.fixar_seed()
dataset, label_names = ner.carregar_dados()

tokenizer = AutoTokenizer.from_pretrained(ner.CHECKPOINT)
dataset_tokenizado = ner.tokenizar_dataset(dataset, tokenizer)

In [ ]:
trainer = ner.treinar_ner(dataset_tokenizado, tokenizer, label_names)

In [ ]:
pred = ner.prever(trainer, dataset_tokenizado, label_names)   # split='test'

## 2. Mismatches (Type-1 a Type-5) — Nejadgholi et al. (2020)

In [ ]:
from metricas import classificar_corpus

todos_detalhes = classificar_corpus(dataset["test"]["tokens"], pred["true_labels"], pred["true_predictions"])

## 3. Automatic Judge

In [ ]:
from juiz import Juiz

juiz = Juiz(dataset, label_names)
juiz.treinar()

In [ ]:
resultados_juiz = juiz.avaliar_erros(todos_detalhes)

## 4. F1: estrito, relaxado, learning-based e estendido

In [ ]:
from metricas import calcular_f1s

f1s = calcular_f1s(todos_detalhes, pred["true_labels"], pred["true_predictions"], resultados_juiz)
print(f"F1 estrito (seqeval): {pred['seqeval']['overall_f1']:.3f}")

## 5. Métricas de ranking — Hasan & Ng (2014)

Score do ranking = confiança do NER por span (`agregacao='media'` ou `'minimo'`).

In [ ]:
from metricas import avaliar_ranking

ranking = avaliar_ranking(dataset["test"]["tokens"], pred, agregacao="media")

## 6. Exemplos reais de cada tipo

In [ ]:
from metricas import mostrar_exemplos

mostrar_exemplos(todos_detalhes, resultados_juiz)